In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [36]:
RAW_DIR = "../data/wisdm-dataset/raw"
SUBJECTS = range(1600, 1651)  # 51 subjects
SENSORS = [("phone", "accel"), ("phone", "gyro"), ("watch", "accel"), ("watch", "gyro")]

In [37]:
def get_sensor_data(filepath):
    df = pd.read_csv(
        filepath,
        header=None,
        names=["subject_id", "activity", "timeStamp", "x", "y", "z"],
    )

    df.iloc[:, -1] = df.iloc[:, -1].str.rstrip(";")

    df = df.astype({"z": float})

    df["timeStamp"] = pd.to_datetime(df["timeStamp"], unit="ns")
    df.drop(columns="subject_id", inplace=True)

    return df

In [58]:
def get_subject_data(subject_id):

    dfs: list[pd.DataFrame] = []

    for device, sensor in SENSORS:

        df = get_sensor_data(
            f"{RAW_DIR}/{device}/{sensor}/data_{subject_id}_{sensor}_{device}.txt"
        )

        suffix = f"{device}_{sensor}"

        df["device"] = device
        df["sensor"] = sensor

        dfs.append(df)

    merged = dfs[0]

    for df in dfs[1:]:
        other = df

        merged = pd.concat([merged, other])

    merged = merged.iloc[:, [5, 6, 0, 1, 2, 3, 4]]

    return merged.sort_values(by=["device", "sensor", "activity", "timeStamp"])

In [59]:
df = get_subject_data(1600)

df

,device,sensor,activity,timeStamp,x,y,z
0,phone,accel,A,1970-01-03 22:03:27.666810782,-0.364761,8.793503,1.055084
1,phone,accel,A,1970-01-03 22:03:27.717164786,-0.879730,9.768784,1.016998
2,phone,accel,A,1970-01-03 22:03:27.767518790,2.001495,11.109070,2.619156
3,phone,accel,A,1970-01-03 22:03:27.817872794,0.450623,12.651642,0.184555
4,phone,accel,A,1970-01-03 22:03:27.868226798,-2.164352,13.928436,-4.422485
...,...,...,...,...,...,...,...
65430,watch,gyro,S,1970-01-01 22:58:44.282947172,1.561949,1.489680,-0.293027
65431,watch,gyro,S,1970-01-01 22:58:44.332877212,0.464727,2.014856,-0.306876
65432,watch,gyro,S,1970-01-01 22:58:44.382807252,0.862081,2.303537,-0.596626
65433,watch,gyro,S,1970-01-01 22:58:44.432737292,0.086569,2.057461,-0.931119
